# Titanic iteration 2: regularization and tree comparison
Select using the mean of stratified and ticket-group CV, before seeing the new leaderboard result.
train.py SHA256 3cc6e2d527cd78281e57461be0767b86fa036dced0bfb7dea44abf040317f4cd; train_v2.py SHA256 128834aa0d221e92ae6d0a452b6295d5ed7c7919230a27f038033a0c6ba3098a.


In [ ]:
from pathlib import Path
Path("/kaggle/working/train.py").write_text('"""Titanic: row-level features, honest CV, and a reproducible submission.\n\nOnly the official competition training labels are used. No external passenger\nrecords, test labels, target encodings, or leaderboard-driven model selection.\n"""\n\nimport argparse\nimport hashlib\nimport json\nimport platform\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport catboost\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom catboost import CatBoostClassifier\nfrom sklearn.metrics import accuracy_score, log_loss\nfrom sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold\n\n\nCATEGORICAL = ["Sex", "Embarked", "Title", "Deck", "TicketPrefix", "ClassSex"]\nCONFIGS = {\n    "catboost_depth4": {"depth": 4, "iterations": 1000, "learning_rate": 0.035, "l2_leaf_reg": 5},\n    "catboost_depth5": {"depth": 5, "iterations": 1000, "learning_rate": 0.035, "l2_leaf_reg": 5},\n    "catboost_depth6": {"depth": 6, "iterations": 1000, "learning_rate": 0.035, "l2_leaf_reg": 8},\n}\n\n\ndef features(df):\n    """Deterministic per-row transforms; never inspect Survived or other rows."""\n    x = df[["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]].copy()\n    x["FamilySize"] = df["SibSp"] + df["Parch"] + 1\n    x["IsAlone"] = (x["FamilySize"] == 1).astype(int)\n    x["FarePerPerson"] = df["Fare"] / x["FamilySize"]\n    x["LogFare"] = np.log1p(df["Fare"])\n    x["AgeMissing"] = df["Age"].isna().astype(int)\n    x["IsChild"] = (df["Age"] < 16).astype(int)\n    x["Title"] = df["Name"].str.extract(r",\\s*([^.]*)\\.", expand=False).str.strip()\n    x["NameLength"] = df["Name"].str.len()\n    cabin = df["Cabin"].fillna("")\n    x["Deck"] = cabin.str.extract(r"([A-Za-z])", expand=False).fillna("Unknown")\n    x["CabinCount"] = cabin.str.split().str.len()\n    x["TicketPrefix"] = df["Ticket"].str.replace(r"[0-9. /]", "", regex=True).str.upper().replace("", "NUMERIC")\n    x["ClassSex"] = df["Pclass"].astype(str) + "_" + df["Sex"]\n    for col in CATEGORICAL:\n        x[col] = x[col].fillna("Unknown").astype(str)\n    # A fixed missing sentinel does not estimate statistics from validation data.\n    for col in x.columns.difference(CATEGORICAL):\n        x[col] = x[col].fillna(-1).astype(float)\n    assert "Survived" not in x and "PassengerId" not in x\n    assert not x.isna().any().any()\n    return x\n\n\ndef model(config, seed):\n    return CatBoostClassifier(**config, loss_function="Logloss", random_seed=seed,\n                              verbose=False, thread_count=2, allow_writing_files=False)\n\n\ndef evaluate(x, y, splits, config):\n    oof = np.zeros(len(y))\n    folds = []\n    for fold, (fit_idx, val_idx) in enumerate(splits):\n        estimator = model(config, 42 + fold)\n        estimator.fit(x.iloc[fit_idx], y.iloc[fit_idx], cat_features=CATEGORICAL)\n        oof[val_idx] = estimator.predict_proba(x.iloc[val_idx])[:, 1]\n        folds.append(float(accuracy_score(y.iloc[val_idx], oof[val_idx] >= 0.5)))\n    return {"accuracy": float(accuracy_score(y, oof >= 0.5)),\n            "log_loss": float(log_loss(y, oof)), "fold_accuracy": folds}, oof\n\n\ndef run(data_dir, output_dir):\n    started = time.time()\n    data_dir, output_dir = Path(data_dir), Path(output_dir)\n    if not (data_dir / "train.csv").exists() and str(data_dir).startswith("/kaggle/input/"):\n        # Kaggle may mount competitions under /kaggle/input/competitions/.\n        candidates = [p.parent for p in Path("/kaggle/input").rglob("train.csv")\n                      if (p.parent / "test.csv").exists() and p.parent.name == "titanic"]\n        assert len(candidates) == 1, f"Expected one official Titanic input: {candidates}"\n        data_dir = candidates[0]\n    print(f"DATA_DIRECTORY={data_dir}", flush=True)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    train, test = pd.read_csv(data_dir / "train.csv"), pd.read_csv(data_dir / "test.csv")\n    assert len(train) == 891 and len(test) == 418\n    assert "Survived" in train and "Survived" not in test\n    assert train.PassengerId.is_unique and test.PassengerId.is_unique\n    assert set(train.Survived.unique()) == {0, 1}\n    x, xt, y = features(train), features(test), train.Survived\n    splits = list(StratifiedKFold(5, shuffle=True, random_state=42).split(x, y))\n    baseline = float(accuracy_score(y, train.Sex.eq("female").astype(int)))\n    scores = {}\n    for name, config in CONFIGS.items():\n        scores[name], _ = evaluate(x, y, splits, config)\n        print(json.dumps({"candidate": name, **scores[name]}), flush=True)\n    selected = max(scores, key=lambda k: (scores[k]["accuracy"], -scores[k]["log_loss"]))\n    # Diagnostic only: prevent shared tickets crossing folds. Model selection\n    # uses the predeclared stratified folds above, not this or the leaderboard.\n    groups = train.Ticket.fillna("MISSING").astype(str)\n    group_splits = list(StratifiedGroupKFold(5, shuffle=True, random_state=42).split(x, y, groups))\n    for fit_idx, val_idx in group_splits:\n        assert set(groups.iloc[fit_idx]).isdisjoint(set(groups.iloc[val_idx]))\n    group_score, _ = evaluate(x, y, group_splits, CONFIGS[selected])\n    probabilities = []\n    for seed in [42, 137, 2026]:\n        estimator = model(CONFIGS[selected], seed)\n        estimator.fit(x, y, cat_features=CATEGORICAL)\n        probabilities.append(estimator.predict_proba(xt)[:, 1])\n    predictions = (np.mean(probabilities, axis=0) >= 0.5).astype(int)\n    submission = pd.DataFrame({"PassengerId": test.PassengerId, "Survived": predictions})\n    assert submission.columns.tolist() == ["PassengerId", "Survived"]\n    assert len(submission) == 418 and submission.PassengerId.is_unique\n    assert submission.Survived.isin([0, 1]).all()\n    assert submission.PassengerId.equals(test.PassengerId)\n    submission_path = output_dir / "submission.csv"\n    submission.to_csv(submission_path, index=False)\n    metrics = {\n        "competition": "titanic", "created_at_utc": datetime.now(timezone.utc).isoformat(),\n        "input_directory": str(data_dir),\n        "train_rows": len(train), "test_rows": len(test),\n        "data_sha256": {name: hashlib.sha256((data_dir/name).read_bytes()).hexdigest()\n                        for name in ["train.csv", "test.csv"]},\n        "versions": {"python": platform.python_version(), "numpy": np.__version__,\n                     "pandas": pd.__version__, "sklearn": sklearn.__version__, "catboost": catboost.__version__},\n        "validation": "5-fold stratified CV; shuffle=True, seed=42; threshold=0.5",\n        "baseline_female_survives_accuracy": baseline,\n        "candidate_scores": scores, "selected_model": selected, "selected_config": CONFIGS[selected],\n        "ticket_group_cv_diagnostic": group_score,\n        "validation_caveat": "Candidate selection on the same folds is optimistic; ticket-group CV is a diagnostic, not an independent untouched holdout. The final three-seed refit ensemble is not separately CV-scored.",\n        "features": x.columns.tolist(), "final_seeds": [42, 137, 2026],\n        "submission_sha256": hashlib.sha256(submission_path.read_bytes()).hexdigest(),\n        "predicted_survivors": int(predictions.sum()), "runtime_seconds": time.time()-started,\n        "kaggle_public_score": None,\n    }\n    (output_dir / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\\n")\n    print("FINAL_METRICS=" + json.dumps(metrics), flush=True)\n    print("Submission verified: 418 unique PassengerIds; binary predictions; exact schema.", flush=True)\n    return metrics\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data-dir", default="/kaggle/input/titanic")\n    parser.add_argument("--output-dir", default="/kaggle/working")\n    args = parser.parse_args()\n    run(args.data_dir, args.output_dir)\n')


In [ ]:
"""A predeclared comparison after v1's validation/leaderboard gap.

Selection uses training-label CV only: average stratified and ticket-group
accuracy. The v1 leaderboard score is not an input to this program.
"""
import argparse
import hashlib
import json
import platform
import time
from datetime import datetime, timezone
from pathlib import Path

import catboost
import numpy as np
import pandas as pd
import sklearn
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from train import CATEGORICAL, features

NAMES = ["catboost_regularized", "random_forest", "extra_trees"]
BASELINE = {"stratified": 0.8383838383838383, "ticket_group": 0.8103254769921436}


def estimator(name, seed, columns):
    if name == "catboost_regularized":
        return CatBoostClassifier(depth=4, iterations=600, learning_rate=0.035,
                                 l2_leaf_reg=15, loss_function="Logloss", random_seed=seed,
                                 thread_count=2, verbose=False, allow_writing_files=False)
    preprocessing = ColumnTransformer([
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
        ("numeric", "passthrough", [c for c in columns if c not in CATEGORICAL]),
    ])
    cls = RandomForestClassifier if name == "random_forest" else ExtraTreesClassifier
    trees = cls(n_estimators=400, max_depth=7, min_samples_leaf=3, max_features=0.8,
                random_state=seed, n_jobs=2)
    return make_pipeline(preprocessing, trees)


def fit(model, name, x, y):
    if name == "catboost_regularized":
        model.fit(x, y, cat_features=CATEGORICAL)
    else:
        model.fit(x, y)
    return model


def cross_validate(name, x, y, splits):
    predictions = np.zeros(len(y))
    scores = []
    for fold, (tr, va) in enumerate(splits):
        model = fit(estimator(name, 42 + fold, x.columns), name, x.iloc[tr], y.iloc[tr])
        predictions[va] = model.predict_proba(x.iloc[va])[:, 1]
        scores.append(float(accuracy_score(y.iloc[va], predictions[va] >= 0.5)))
    return {"accuracy": float(accuracy_score(y, predictions >= 0.5)),
            "log_loss": float(log_loss(y, predictions)), "fold_accuracy": scores}


def run_v2(data_dir, output_dir):
    start = time.time()
    data_dir, out = Path(data_dir), Path(output_dir)
    if not (data_dir / "train.csv").exists():
        candidates = [p.parent for p in Path("/kaggle/input").rglob("train.csv")
                      if p.parent.name == "titanic" and (p.parent / "test.csv").exists()]
        assert len(candidates) == 1
        data_dir = candidates[0]
    out.mkdir(parents=True, exist_ok=True)
    train, test = pd.read_csv(data_dir/"train.csv"), pd.read_csv(data_dir/"test.csv")
    assert len(train) == 891 and len(test) == 418 and "Survived" not in test
    # Remove name length: it has no clear survival mechanism and can overfit.
    x, xt, y = features(train).drop(columns="NameLength"), features(test).drop(columns="NameLength"), train.Survived
    groups = train.Ticket.fillna("MISSING").astype(str)
    stratified = list(StratifiedKFold(5, shuffle=True, random_state=42).split(x, y))
    grouped = list(StratifiedGroupKFold(5, shuffle=True, random_state=42).split(x, y, groups))
    for tr, va in grouped:
        assert set(groups.iloc[tr]).isdisjoint(set(groups.iloc[va]))
    scores = {}
    for name in NAMES:
        scores[name] = {"stratified": cross_validate(name, x, y, stratified),
                        "ticket_group": cross_validate(name, x, y, grouped)}
        scores[name]["selection_score"] = (scores[name]["stratified"]["accuracy"] + scores[name]["ticket_group"]["accuracy"]) / 2
        print("CANDIDATE=" + json.dumps({"name": name, **scores[name]}), flush=True)
    selected = max(NAMES, key=lambda name: scores[name]["selection_score"])
    baseline_score = (BASELINE["stratified"] + BASELINE["ticket_group"])/2
    # Freeze the submission gate before learning the new leaderboard result.
    worth_submitting = (scores[selected]["selection_score"] >= baseline_score + 0.002
                        and scores[selected]["ticket_group"]["accuracy"] >= BASELINE["ticket_group"] + 0.005)
    probabilities = []
    for seed in [42, 137, 2026]:
        model = fit(estimator(selected, seed, x.columns), selected, x, y)
        probabilities.append(model.predict_proba(xt)[:, 1])
    predictions = (np.mean(probabilities, axis=0) >= 0.5).astype(int)
    submission = pd.DataFrame({"PassengerId": test.PassengerId, "Survived": predictions})
    assert len(submission) == 418 and submission.PassengerId.is_unique
    assert submission.Survived.isin([0, 1]).all()
    assert submission.PassengerId.equals(test.PassengerId)
    path = out/"submission.csv"
    submission.to_csv(path, index=False)
    metrics = {
        "competition": "titanic", "iteration": 2,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "data_sha256": {f: hashlib.sha256((data_dir/f).read_bytes()).hexdigest() for f in ["train.csv", "test.csv"]},
        "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__, "catboost": catboost.__version__},
        "candidate_scores": scores, "selected_model": selected,
        "selection_rule": "Maximize mean of fixed stratified CV and ticket-group CV accuracy; threshold=0.5.",
        "baseline_cv": BASELINE, "baseline_selection_score": baseline_score,
        "submission_gate": "mean CV improves >=0.002 and ticket-group CV improves >=0.005 versus v1",
        "worth_submitting": bool(worth_submitting), "features": x.columns.tolist(), "final_seeds": [42, 137, 2026],
        "submission_sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        "predicted_survivors": int(predictions.sum()), "runtime_seconds": time.time()-start,
        "caveat": "Both CV schemes have been reused for candidate selection, so they are optimistic. The final three-seed refit ensemble is not separately CV-scored. No external or test labels are used.",
    }
    (out/"metrics_v2.json").write_text(json.dumps(metrics, indent=2)+"\n")
    print("FINAL_METRICS_V2=" + json.dumps(metrics), flush=True)
    print("Submission verified: 418 unique PassengerIds; binary predictions; exact schema.", flush=True)
    return metrics


if False:
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", default="/kaggle/input/titanic")
    parser.add_argument("--output-dir", default="/kaggle/working")
    args = parser.parse_args()
    run_v2(args.data_dir, args.output_dir)

metrics = run_v2("/kaggle/input/titanic", "/kaggle/working")
